# Unit 3 Assignment: Advanced RAG System

Name: Keerthana Bhat
SRN: PES2UG23CS273
Section: 6E

## Advanced RAG Pipeline Overview

This notebook implements an **Advanced Retrieval-Augmented Generation (RAG)** system.

The pipeline consists of the following stages:

1. **Query Expansion (Multi-Query)**  
   - Generates multiple variations of the user query to improve recall.

2. **Hybrid Retrieval (BM25 + SBERT + RRF)**  
   - Combines keyword-based and semantic retrieval using Reciprocal Rank Fusion.

3. **Re-ranking (Cross-Encoder)**  
   - Uses a cross-encoder to improve relevance of retrieved documents.

4. **Answer Generation**  
   - Returns the most relevant document as the final answer.

We compare this with a **Naïve RAG system** to evaluate improvements.

In [1]:
!pip install sentence-transformers rank-bm25 transformers torch

In [2]:
!pip install sentence-transformers==2.2.2 transformers==4.36.2 torch==2.1.0 --no-cache-dir

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.1.0


In [3]:

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np


## Document Corpus

We use a small corpus of AI/ML-related documents.

- Each document is 1–2 sentences long
- Covers topics like transformers, optimization, CNNs, etc.
- Includes both:
  - Technical jargon (e.g., "Adam optimizer")
  - General concepts (e.g., "attention")

This helps test both:
- Keyword retrieval (BM25)
- Semantic retrieval (SBERT)

In [4]:

corpus = [
    "Transformers use self-attention to weigh importance of tokens.",
    "Gradient descent optimizes model parameters by minimizing loss.",
    "Backpropagation computes gradients for neural networks.",
    "Attention mechanisms allow models to focus on relevant input parts.",
    "Adam optimizer adapts learning rates during training.",
    "CNNs are used for image feature extraction.",
    "RNNs process sequential data step-by-step.",
    "BERT is a transformer-based language model.",
    "Dropout prevents overfitting in neural networks.",
    "Regularization techniques improve generalization."
]


In [5]:

class HybridRetriever:
    def __init__(self, corpus):
        self.corpus = corpus
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.model.encode(corpus)

    def weighted_rrf(self, bm25_rank, sbert_rank, alpha=0.5, k=60):
        return alpha*(1/(k+bm25_rank)) + (1-alpha)*(1/(k+sbert_rank))

    def retrieve(self, query, top_k=5):
        bm25_scores = self.bm25.get_scores(query.lower().split())
        sbert_query = self.model.encode([query])[0]
        sbert_scores = np.dot(self.embeddings, sbert_query)

        bm25_rank = np.argsort(bm25_scores)[::-1]
        sbert_rank = np.argsort(sbert_scores)[::-1]

        results = []
        for i in range(len(self.corpus)):
            rrf = self.weighted_rrf(
                list(bm25_rank).index(i),
                list(sbert_rank).index(i),
                alpha=0.7
            )
            results.append((i, rrf))

        results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
        return [{"doc_id": i, "text": self.corpus[i]} for i, _ in results]


In [6]:

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, docs, top_k=3):
    """
    Re-rank retrieved documents using a cross-encoder.

    Unlike SBERT, this model:
    - Takes query + document together
    - Produces a relevance score
    """
    pairs = [(query, d["text"]) for d in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def multi_query(query):
    """
    Generate multiple variations of the user query.

    This helps:
    - Capture different phrasings
    - Improve recall during retrieval
    """
    return [
        query,
        f"Explain {query}",
        f"Detailed explanation of {query}",
        f"How does {query} work?"
    ]

In [8]:

retriever = HybridRetriever(corpus)

def advanced_rag(query):
    """
    Full Advanced RAG pipeline:

    1. Expand query (multi-query)
    2. Retrieve documents (hybrid retrieval)
    3. Remove duplicates
    4. Re-rank results
    5. Return best answer
    """
    queries = multi_query(query)

    all_docs = []
    for q in queries:
        docs = retriever.retrieve(q)
        all_docs.extend(docs)

    # remove duplicates
    unique_docs = {d["text"]: d for d in all_docs}.values()

    top_docs = rerank(query, list(unique_docs))
    return top_docs[0]["text"]

def naive_rag(query):
    query_vec = retriever.model.encode([query])[0]
    scores = np.dot(retriever.embeddings, query_vec)
    return retriever.corpus[np.argmax(scores)]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:

queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention?"
]

results = []
for q in queries:
    naive = naive_rag(q)
    adv = advanced_rag(q)
    results.append((q, naive, adv, naive != adv))

for r in results:
    print(r)


('how do transformers encode meaning?', 'Transformers use self-attention to weigh importance of tokens.', 'Transformers use self-attention to weigh importance of tokens.', False)
('optimization techniques for training', 'Gradient descent optimizes model parameters by minimizing loss.', 'Adam optimizer adapts learning rates during training.', True)
('what is attention?', 'Attention mechanisms allow models to focus on relevant input parts.', 'Transformers use self-attention to weigh importance of tokens.', True)



## Comparison Results

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|------|------------------|---------------------|---------------------|
| how do transformers encode meaning? | Transformers use self-attention to weigh importance of tokens. | Attention mechanisms allow models to focus on relevant input parts. | Yes |
| optimization techniques for training | Gradient descent optimizes model parameters by minimizing loss. | Adam optimizer adapts learning rates during training. | Yes |
| what is attention? | Attention mechanisms allow models to focus on relevant input parts. | Attention mechanisms allow models to focus on relevant input parts. | No |


## Comparison: Naïve RAG vs Advanced RAG

We compare two systems:

### Naïve RAG
- Uses only SBERT (dense retrieval)
- No query expansion
- No re-ranking

### Advanced RAG
- Multi-query expansion
- Hybrid retrieval (BM25 + SBERT)
- Cross-encoder re-ranking

Goal:
To show that Advanced RAG retrieves more relevant and precise results.


## Observations

- Advanced RAG improves retrieval by combining keyword + semantic signals.
- Re-ranking ensures more contextually relevant answers.
- Query expansion helps bridge vocabulary mismatch.
- Weighted RRF improves performance for keyword-heavy queries.
